# Test Gemma 3 4B IT OCR correction
Use godfather.venv. Restart the kernel to release ByT5, then run the load cell once. Edit text in the last cell and run it to print the correction only.
Loads C:/MODELS/gemma-3-4b-it locally in 4-bit on CUDA; no downloads or pipeline changes.


In [5]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration, BitsAndBytesConfig

MODEL_PATH = Path('C:/MODELS/gemma-3-4b-it')
assert (MODEL_PATH / 'config.json').is_file(), f'Missing local model: {MODEL_PATH}'
assert torch.cuda.is_available(), 'Select your CUDA godfather.venv kernel.'
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = Gemma3ForConditionalGeneration.from_pretrained(
    str(MODEL_PATH), local_files_only=True, device_map='auto',
    dtype=compute_dtype,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                         bnb_4bit_compute_dtype=compute_dtype),
).eval()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
# text = 'IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW HER. IF SHE SUCCEEDS IN LURING OUT ONE PERSON IN THIS PLACE,'
text = '444'
# Examples show OCR-noise removal, rather than copying the noisy input.
messages = [
    {'role': 'system', 'content':
     'You restore English manga dialogue corrupted by OCR. '
     'Remove nonsensical inserted letter sequences and accidental isolated letters that interrupt a sentence. '
     'Repair obvious broken words. Preserve legitimate names, valid single-letter words such as I and A, '
     'capitalization, meaning, punctuation and unfinished sentences. '
     'Do not rewrite the dialogue, complete it or answer it. '
     'Return only the restored text, with no explanation or quotation marks.'},
    {'role': 'user', 'content': 'PLEASE Q OPEN THE WINDOW.'},
    {'role': 'assistant', 'content': 'PLEASE OPEN THE WINDOW.'},
    {'role': 'user', 'content': 'I CAN HEAR XZQWZXQ SOMEONE OUTSIDE...'},
    {'role': 'assistant', 'content': 'I CAN HEAR SOMEONE OUTSIDE...'},
    {'role': 'user', 'content': 'AH... SHIROGANE, WAIT!'},
    {'role': 'assistant', 'content': 'AH... SHIROGANE, WAIT!'},
    {'role': 'user', 'content': text},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                       tokenize=True, return_dict=True, return_tensors='pt').to(model.device)
with torch.inference_mode():
    output = model.generate(**inputs, do_sample=False,
                            max_new_tokens=max(256, inputs['input_ids'].shape[1] * 2))
print(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip())


444


In [37]:
# text = 'IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW HER. IF SHE SUCCEEDS IN LURING OUT ONE PERSON IN THIS PLACE,'
text = '!!!!'
# Examples show OCR-noise removal, rather than copying the noisy input.
messages = [
    {'role': 'system', 'content':
     'You determin Text is sensical English Dialogue in a conversation or some wierd artifact getting transcripted wrongly '
     'Return Text for correct Text or japanse stuff, Artifact for wierd stuff'
     'Punctuation only sentences are Text'
     'the text given might be japanse name or noun, only judge on basis of artifact data not the semantic'
     'also reply artifact for SFX or visual sounds in text form'},
    {'role': 'user', 'content': 'oo'},
    {'role': 'assistant', 'content': 'artifact'},
    {'role': 'user', 'content': 'www.google.com'},
    {'role': 'assistant', 'content': 'artifact'},
    {'role': 'user', 'content': 'Stream Do not mirror'},
    {'role': 'assistant', 'content': 'Artifact'},
    {'role': 'user', 'content': 'I said to not do that '},
    {'role': 'assistant', 'content': 'text'},
    {'role': 'user', 'content': 'But our characters were unaware'},
    {'role': 'assistant', 'content': 'text'},
    {'role': 'user', 'content': 'Whoosh!!'},
    {'role': 'assistant', 'content': 'artifact'},
    {'role': 'user', 'content': 'BAM'},
    {'role': 'assistant', 'content': 'artifact'},
    {'role': 'user', 'content': 'NANIKAWA'},
    {'role': 'assistant', 'content': 'text'},
    {'role': 'user', 'content': '...'},
    {'role': 'assistant', 'content': 'text'},
    {'role': 'user', 'content': '...!!!'},
    {'role': 'assistant', 'content': 'text'},
    {'role': 'user', 'content': '!'},
    {'role': 'assistant', 'content': 'text'},
    {'role': 'user', 'content': text},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                       tokenize=True, return_dict=True, return_tensors='pt').to(model.device)
with torch.inference_mode():
    output = model.generate(**inputs, do_sample=False,
                            max_new_tokens=max(256, inputs['input_ids'].shape[1] * 2))
print(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip())


artifact
